<a href="https://colab.research.google.com/github/trungkien1502/project_sna_group14/blob/main/01_data_inspection.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
from google.colab import drive
from pathlib import Path

drive.mount('/content/drive')

root = Path('/content/drive/MyDrive')
required = [
    'trn_mat.pkl',
    'val_mat.pkl',
    'tst_mat.pkl',
    'usr_prf.pkl',
    'itm_prf.pkl',
    'usr_emb_np.pkl',
    'itm_emb_np.pkl'
]

folders = [
    p.parent
    for p in root.rglob('trn_mat.pkl')
    if p.parent.name == 'amazon'
]

if not folders:
    print("Chưa tìm thấy thư mục amazon chứa trn_mat.pkl")
else:
    folder = folders[0]
    print("Dataset:", folder)

    for name in required:
        path = folder / name
        if path.exists():
            print(f"OK      {name} | {path.stat().st_size / 1024**2:.2f} MB")
        else:
            print(f"MISSING {name}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Dataset: /content/drive/MyDrive/Data_SNA_Group14/data/raw/amazon
OK      trn_mat.pkl | 1.84 MB
OK      val_mat.pkl | 0.62 MB
OK      tst_mat.pkl | 0.61 MB
OK      usr_prf.pkl | 9.58 MB
OK      itm_prf.pkl | 8.05 MB
OK      usr_emb_np.pkl | 128.91 MB
OK      itm_emb_np.pkl | 109.36 MB


In [3]:
import pickle
import json
import numpy as np
from scipy import sparse
from pathlib import Path

DATA_DIR = Path(
    "/content/drive/MyDrive/Data_SNA_Group14/data/raw/amazon"
)

# Chỉ mở pickle từ nguồn tác giả đã tin cậy.
def load_pkl(name):
    with open(DATA_DIR / name, "rb") as f:
        return pickle.load(f)

# 1. Kiểm tra train/validation/test
mats = {}

for split, filename in [
    ("train", "trn_mat.pkl"),
    ("validation", "val_mat.pkl"),
    ("test", "tst_mat.pkl")
]:
    mat = load_pkl(filename)
    assert sparse.issparse(mat), f"{filename} không phải sparse matrix"

    mat = (mat.tocsr() != 0).astype(np.int8)
    mats[split] = mat

    print(f"{split}: shape={mat.shape}, interactions={mat.nnz}")

# 2. Kiểm tra kích thước
assert len({m.shape for m in mats.values()}) == 1

users, items = mats["train"].shape
print(f"\nUsers: {users}")
print(f"Items: {items}")
print(f"Total nodes: {users + items}")

# 3. Kiểm tra trùng giữa các split
print("\nSPLIT OVERLAP")

overlaps = {}
for a, b in [
    ("train", "validation"),
    ("train", "test"),
    ("validation", "test")
]:
    n = mats[a].multiply(mats[b]).count_nonzero()
    overlaps[f"{a}_{b}"] = int(n)
    print(f"{a} vs {b}: {n}")

# 4. Kiểm tra profiles và embeddings
print("\nSEMANTIC DATA")

semantic_info = {}

for name in [
    "usr_prf.pkl",
    "itm_prf.pkl",
    "usr_emb_np.pkl",
    "itm_emb_np.pkl"
]:
    obj = load_pkl(name)
    arr = np.asarray(obj) if not isinstance(obj, dict) else None

    info = {
        "type": type(obj).__name__,
        "shape": list(arr.shape) if arr is not None else None,
        "length": len(obj)
    }

    if arr is not None and np.issubdtype(arr.dtype, np.number):
        info["all_finite"] = bool(np.isfinite(arr).all())

    semantic_info[name] = info
    print(name, info)

# 5. Lưu báo cáo
report = {
    "users": users,
    "items": items,
    "interactions": {k: int(v.nnz) for k, v in mats.items()},
    "overlaps": overlaps,
    "semantic": semantic_info
}

output_dir = DATA_DIR.parent.parent / "manifests"
output_dir.mkdir(parents=True, exist_ok=True)

output_file = output_dir / "dataset_inspection.json"
output_file.write_text(json.dumps(report, indent=2))

print("\nReport saved:", output_file)

train: shape=(11000, 9332), interactions=120464
validation: shape=(11000, 9332), interactions=40290
test: shape=(11000, 9332), interactions=40106

Users: 11000
Items: 9332
Total nodes: 20332

SPLIT OVERLAP
train vs validation: 0
train vs test: 0
validation vs test: 0

SEMANTIC DATA
usr_prf.pkl {'type': 'dict', 'shape': None, 'length': 11000}
itm_prf.pkl {'type': 'dict', 'shape': None, 'length': 9332}
usr_emb_np.pkl {'type': 'ndarray', 'shape': [11000, 1536], 'length': 11000, 'all_finite': True}
itm_emb_np.pkl {'type': 'ndarray', 'shape': [9332, 1536], 'length': 9332, 'all_finite': True}

Report saved: /content/drive/MyDrive/Data_SNA_Group14/data/manifests/dataset_inspection.json
